<a href="https://colab.research.google.com/github/frazigg02/Neural_collapse-/blob/main/notebooks/E-7bis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

E-7. Regressione logistica e random forest sulle 48 statistiche, con il protocollo del §4.4, per K = 2 e K = 20. Riportiamo F1 pesato e matrice di
confusione.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

percorso = "/content/drive/MyDrive/Neural Collapse/USTC-TFC2016/processed/ustc_tfc2016_preprocessed.parquet"

print("Dataset trovato:", os.path.exists(percorso))

Dataset trovato: True


In [ ]:
import pandas as pd
import numpy as np

E6_PATH = (
    "/content/drive/MyDrive/Neural Collapse/"
    "USTC-TFC2016/processed/ustc_tfc2016_preprocessed.parquet"
)

print("Caricamento dataset E-6...")

df = pd.read_parquet(E6_PATH)

print("Dataset caricato!")
print("Flussi:", len(df))
print("Colonne:", len(df.columns))

print("\nDistribuzione split:")
print(df["split"].value_counts())

print("\nClassi label_20:", df["label_20"].nunique())
print("Classi label_2:", df["label_2"].nunique())

Caricamento dataset E-6...
Dataset caricato!
Flussi: 562549
Colonne: 66

Distribuzione split:
split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64

Classi label_20: 20
Classi label_2: 2


In [ ]:
df_final = df

print("df_final pronto:", df_final.shape)

df_final pronto: (562549, 66)


In [ ]:
# E-7 — Creazione delle 48 statistiche aggregate

import numpy as np

print("Creazione matrice delle 48 statistiche...")

stats_48 = np.column_stack([

    # DURATA / CONTEGGI / BYTES — 9
    df_final["bidirectional_duration_ms"].to_numpy(),
    df_final["bidirectional_packets"].to_numpy(),
    df_final["bidirectional_bytes"].to_numpy(),

    df_final["src2dst_duration_ms"].to_numpy(),
    df_final["src2dst_packets"].to_numpy(),
    df_final["src2dst_bytes"].to_numpy(),

    df_final["dst2src_duration_ms"].to_numpy(),
    df_final["dst2src_packets"].to_numpy(),
    df_final["dst2src_bytes"].to_numpy(),

    # PACKET SIZE — 12
    df_final["bidirectional_min_ps"].to_numpy(),
    df_final["bidirectional_mean_ps"].to_numpy(),
    df_final["bidirectional_stddev_ps"].to_numpy(),
    df_final["bidirectional_max_ps"].to_numpy(),

    df_final["src2dst_min_ps"].to_numpy(),
    df_final["src2dst_mean_ps"].to_numpy(),
    df_final["src2dst_stddev_ps"].to_numpy(),
    df_final["src2dst_max_ps"].to_numpy(),

    df_final["dst2src_min_ps"].to_numpy(),
    df_final["dst2src_mean_ps"].to_numpy(),
    df_final["dst2src_stddev_ps"].to_numpy(),
    df_final["dst2src_max_ps"].to_numpy(),

    # PIAT — 12
    df_final["bidirectional_min_piat_ms"].to_numpy(),
    df_final["bidirectional_mean_piat_ms"].to_numpy(),
    df_final["bidirectional_stddev_piat_ms"].to_numpy(),
    df_final["bidirectional_max_piat_ms"].to_numpy(),

    df_final["src2dst_min_piat_ms"].to_numpy(),
    df_final["src2dst_mean_piat_ms"].to_numpy(),
    df_final["src2dst_stddev_piat_ms"].to_numpy(),
    df_final["src2dst_max_piat_ms"].to_numpy(),

    df_final["dst2src_min_piat_ms"].to_numpy(),
    df_final["dst2src_mean_piat_ms"].to_numpy(),
    df_final["dst2src_stddev_piat_ms"].to_numpy(),
    df_final["dst2src_max_piat_ms"].to_numpy(),

    # FLAG TCP — 15
    df_final["bidirectional_syn_packets"].to_numpy(),
    df_final["bidirectional_cwr_packets"].to_numpy(),
    df_final["bidirectional_ece_packets"].to_numpy(),
    df_final["bidirectional_urg_packets"].to_numpy(),
    df_final["bidirectional_ack_packets"].to_numpy(),
    df_final["bidirectional_psh_packets"].to_numpy(),
    df_final["bidirectional_rst_packets"].to_numpy(),
    df_final["bidirectional_fin_packets"].to_numpy(),

    df_final["src2dst_syn_packets"].to_numpy(),
    df_final["src2dst_ack_packets"].to_numpy(),
    df_final["src2dst_psh_packets"].to_numpy(),
    df_final["src2dst_rst_packets"].to_numpy(),

    df_final["dst2src_syn_packets"].to_numpy(),
    df_final["dst2src_ack_packets"].to_numpy(),
    df_final["dst2src_psh_packets"].to_numpy()
])

print("Matrice creata!")
print("Dimensioni:", stats_48.shape)

print("\nControllo NaN:", np.isnan(stats_48).sum())
print("Controllo inf:", np.isinf(stats_48).sum())

Creazione matrice delle 48 statistiche...
Matrice creata!
Dimensioni: (562549, 48)

Controllo NaN: 0
Controllo inf: 0


In [ ]:
# PREPARAZIONE DATI E-7 — K = 2

import numpy as np

print("## Preparazione dati K = 2...")

# Le 48 statistiche calcolate sopra
X = stats_48.astype(np.float32)

# Target binario
y = df["label_2"].to_numpy()

# Split già definiti in E-6: NON vengono modificati
split = df["split"].to_numpy()

# Maschere per i quattro split
train_mask = split == "train"
val_mask   = split == "val"
test_mask  = split == "test"
probe_mask = split == "probe"

# Creazione dei dataset
X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]

X_probe = X[probe_mask]
y_probe = y[probe_mask]

print("\nDimensioni:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Probe:", X_probe.shape)

print("\nNumero classi:", len(np.unique(y_train)))

print("\nPreparazione K = 2 completata.")

## Preparazione dati K = 2...

Dimensioni:
Train: (393783, 48)
Validation: (84383, 48)
Test: (56255, 48)
Probe: (28128, 48)

Numero classi: 2

Preparazione K = 2 completata.


In [ ]:

# E-7bis — PREPARAZIONE DATI K = 2 CON 47 FEATURE

X_train_47 = X_47[train_mask]
X_val_47 = X_47[val_mask]
X_test_47 = X_47[test_mask]
X_probe_47 = X_47[probe_mask]

print("Train:", X_train_47.shape)
print("Validation:", X_val_47.shape)
print("Test:", X_test_47.shape)
print("Probe:", X_probe_47.shape)


Train: (393783, 47)
Validation: (84383, 47)
Test: (56255, 47)
Probe: (28128, 47)


In [ ]:
# E-7 — STANDARDIZZAZIONE + REGRESSIONE LOGISTICA K = 2

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

print("Standardizzazione delle 48 statistiche...")

# Lo scaler viene FITTATO SOLO sul TRAIN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Validation, test e probe vengono SOLO trasformati
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_probe_scaled = scaler.transform(X_probe)

print("Standardizzazione completata.")

print("\nAddestramento Regressione Logistica K = 2...")

lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

# Validation
y_val_pred = lr.predict(X_val_scaled)
f1_val = f1_score(
    y_val,
    y_val_pred,
    average="weighted"
)

# Test
y_test_pred = lr.predict(X_test_scaled)
f1_test = f1_score(
    y_test,
    y_test_pred,
    average="weighted"
)

cm_test = confusion_matrix(y_test, y_test_pred)

print("\nREGRESSIONE LOGISTICA K = 2")
print("F1 weighted validation:", f1_val)
print("F1 weighted test:", f1_test)

print("\nMatrice di confusione TEST:")
print(cm_test)

Standardizzazione delle 48 statistiche...
Standardizzazione completata.

Addestramento Regressione Logistica K = 2...

REGRESSIONE LOGISTICA K = 2
F1 weighted validation: 0.9930409256573689
F1 weighted test: 0.9934205227454526

Matrice di confusione TEST:
[[30896    93]
 [  277 24989]]


In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("X_47:", X_47.shape)

X_train: (393783, 48)
X_val: (84383, 48)
X_test: (56255, 48)
X_47: (562549, 47)


In [ ]:
print([nome for nome in globals() if "idx" in nome.lower()])

[]


In [ ]:

# E-7bis — RANDOM FOREST K = 2, 47 FEATURE

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Random Forest K = 2 con 47 feature...")

rf_47 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_47.fit(X_train_47, y_train)

# Validation
y_val_pred_rf_47 = rf_47.predict(X_val_47)

f1_val_rf_47 = f1_score(
    y_val,
    y_val_pred_rf_47,
    average="weighted"
)

# Test
y_test_pred_rf_47 = rf_47.predict(X_test_47)

f1_test_rf_47 = f1_score(
    y_test,
    y_test_pred_rf_47,
    average="weighted"
)

cm_test_rf_47 = confusion_matrix(
    y_test,
    y_test_pred_rf_47
)

print("\nRANDOM FOREST K = 2 — 47 FEATURE")
print("F1 weighted validation:", f1_val_rf_47)
print("F1 weighted test:", f1_test_rf_47)

print("\nMatrice di confusione TEST:")
print(cm_test_rf_47)


Addestramento Random Forest K = 2 con 47 feature...

RANDOM FOREST K = 2 — 47 FEATURE
F1 weighted validation: 0.8528595705230025
F1 weighted test: 0.8549766388482126

Matrice di confusione TEST:
[[ 478    0    0    0    0    0    0    0    0    0    0  273    0    0
     1    0    0    0    0    0]
 [   0 4840    0    0 1298    0    1   30    0    0    0    0    0    1
     0    0    0    0    0    0]
 [   0    0 9889    0    0    0    0    0  215    0    0    0    0    0
     0    0    0    0    0    0]
 [   0    0    0  600    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [   0  628    0    0 4200    0   11  204    0    0    0    0    0    3
     0    0    0    0    0    0]
 [   4    0  202    0    0  596    0    0    0    0    0   56    0    0
     4    0    0    0    1    0]
 [   0    3    0    0   99    0  705    0    0    1    1    0    0    1
     0    1    0    0    0    0]
 [   0   36    0    0  503    0    3  927    0    0    2    0    0    

In [ ]:
# E-7 — RANDOM FOREST K = 2

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Random Forest K = 2...")

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Validation
y_val_pred_rf = rf.predict(X_val)

f1_val_rf = f1_score(
    y_val,
    y_val_pred_rf,
    average="weighted"
)

# Test
y_test_pred_rf = rf.predict(X_test)

f1_test_rf = f1_score(
    y_test,
    y_test_pred_rf,
    average="weighted"
)

cm_test_rf = confusion_matrix(
    y_test,
    y_test_pred_rf
)

print("\nRANDOM FOREST K = 2")
print("F1 weighted validation:", f1_val_rf)
print("F1 weighted test:", f1_test_rf)

print("\nMatrice di confusione TEST:")
print(cm_test_rf)

Addestramento Random Forest K = 2...

RANDOM FOREST K = 2
F1 weighted validation: 0.999976298601282
F1 weighted test: 1.0

Matrice di confusione TEST:
[[30989     0]
 [    0 25266]]


7-E bis:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# Stesso ordine ESATTO con cui ha impilato le colonne in stats_48 (E-7).
feature_names = [
    "bidirectional_duration_ms", "bidirectional_packets", "bidirectional_bytes",
    "src2dst_duration_ms", "src2dst_packets", "src2dst_bytes",
    "dst2src_duration_ms", "dst2src_packets", "dst2src_bytes",

    "bidirectional_min_ps", "bidirectional_mean_ps", "bidirectional_stddev_ps",
    "bidirectional_max_ps",

    "src2dst_min_ps", "src2dst_mean_ps", "src2dst_stddev_ps", "src2dst_max_ps",

    "dst2src_min_ps", "dst2src_mean_ps", "dst2src_stddev_ps", "dst2src_max_ps",

    "bidirectional_min_piat_ms", "bidirectional_mean_piat_ms",
    "bidirectional_stddev_piat_ms", "bidirectional_max_piat_ms",

    "src2dst_min_piat_ms", "src2dst_mean_piat_ms",
    "src2dst_stddev_piat_ms", "src2dst_max_piat_ms",

    "dst2src_min_piat_ms", "dst2src_mean_piat_ms",
    "dst2src_stddev_piat_ms", "dst2src_max_piat_ms",

    "bidirectional_syn_packets", "bidirectional_cwr_packets",
    "bidirectional_ece_packets", "bidirectional_urg_packets",

    "bidirectional_ack_packets", "bidirectional_psh_packets",
    "bidirectional_rst_packets", "bidirectional_fin_packets",

    "src2dst_syn_packets", "src2dst_ack_packets",
    "src2dst_psh_packets", "src2dst_rst_packets",

    "dst2src_syn_packets", "dst2src_ack_packets",
    "dst2src_psh_packets",
]

assert len(feature_names) == 48 == X_train.shape[1]


# ============================================================
# 1) IMPORTANZA DA IMPURITÀ
# ============================================================

imp_impurity = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)


# ============================================================
# 2) PERMUTATION IMPORTANCE
# ============================================================

perm_result = permutation_importance(
    rf,
    X_val,
    y_val,
    scoring="f1_weighted",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

imp_perm = pd.Series(
    perm_result.importances_mean,
    index=feature_names
).sort_values(ascending=False)


print("=== TOP 15 — importanza da impurità (K=2) ===")
print(imp_impurity.head(15))

print()

print("=== TOP 15 — permutation importance su validation (K=2) ===")
print(imp_perm.head(15))


# ============================================================
# 3) DIVISIONE DELLE FEATURE IN DUE GRUPPI
# ============================================================

# Feature di scala grezza:
# durata, numero totale di pacchetti e byte totali del flusso
raw_scale = [
    "bidirectional_duration_ms",
    "bidirectional_packets",
    "bidirectional_bytes",
    "src2dst_duration_ms",
    "src2dst_packets",
    "src2dst_bytes",
    "dst2src_duration_ms",
    "dst2src_packets",
    "dst2src_bytes",
]

# Feature di forma:
# statistiche delle dimensioni dei pacchetti,
# statistiche degli inter-arrivi (PIAT)
# e flag TCP
shape_like = [
    "bidirectional_min_ps",
    "bidirectional_mean_ps",
    "bidirectional_stddev_ps",
    "bidirectional_max_ps",

    "src2dst_min_ps",
    "src2dst_mean_ps",
    "src2dst_stddev_ps",
    "src2dst_max_ps",

    "dst2src_min_ps",
    "dst2src_mean_ps",
    "dst2src_stddev_ps",
    "dst2src_max_ps",

    "bidirectional_min_piat_ms",
    "bidirectional_mean_piat_ms",
    "bidirectional_stddev_piat_ms",
    "bidirectional_max_piat_ms",

    "src2dst_min_piat_ms",
    "src2dst_mean_piat_ms",
    "src2dst_stddev_piat_ms",
    "src2dst_max_piat_ms",

    "dst2src_min_piat_ms",
    "dst2src_mean_piat_ms",
    "dst2src_stddev_piat_ms",
    "dst2src_max_piat_ms",

    "bidirectional_syn_packets",
    "bidirectional_cwr_packets",
    "bidirectional_ece_packets",
    "bidirectional_urg_packets",

    "bidirectional_ack_packets",
    "bidirectional_psh_packets",
    "bidirectional_rst_packets",
    "bidirectional_fin_packets",

    "src2dst_syn_packets",
    "src2dst_ack_packets",
    "src2dst_psh_packets",
    "src2dst_rst_packets",

    "dst2src_syn_packets",
    "dst2src_ack_packets",
    "dst2src_psh_packets",
]


# ============================================================
# 4) CONTROLLI RICHIESTI DALLA TRACCIA
# ============================================================

assert set(raw_scale) | set(shape_like) == set(feature_names)

assert set(raw_scale) & set(shape_like) == set()


print()

print(
    "Somma importanza (impurità) — feature di scala grezza:",
    imp_impurity[raw_scale].sum()
)

print(
    "Somma importanza (impurità) — feature di forma:",
    imp_impurity[shape_like].sum()
)

print()

print(
    "Somma totale importanze:",
    imp_impurity.sum()
)

=== TOP 15 — importanza da impurità (K=2) ===
dst2src_min_ps                  0.127619
bidirectional_mean_piat_ms      0.094841
src2dst_duration_ms             0.088947
bidirectional_duration_ms       0.080651
bidirectional_max_piat_ms       0.071715
bidirectional_min_ps            0.065416
src2dst_min_ps                  0.046299
src2dst_stddev_piat_ms          0.044056
src2dst_mean_piat_ms            0.042712
dst2src_mean_ps                 0.038225
bidirectional_stddev_piat_ms    0.035439
src2dst_max_piat_ms             0.031831
bidirectional_ack_packets       0.030863
src2dst_ack_packets             0.030701
dst2src_ack_packets             0.021638
dtype: float64

=== TOP 15 — permutation importance su validation (K=2) ===
bidirectional_min_ps         0.000872
src2dst_ack_packets          0.000408
bidirectional_ack_packets    0.000014
bidirectional_duration_ms    0.000000
src2dst_packets              0.000000
src2dst_bytes                0.000000
bidirectional_bytes          0.0000

In [ ]:
# PREPARAZIONE DATI K = 20

import numpy as np

print("## Preparazione dati K = 20...")

# Feature: 48 statistiche
# stats_48 è già un numpy.ndarray

X = stats_48.astype(np.float32)

# Target: classificazione a 20 classi

y = df["label_20"].values

# Split già definiti in E-6
# NON vengono modificati

train_mask = df["split"].values == "train"
val_mask   = df["split"].values == "val"
test_mask  = df["split"].values == "test"
probe_mask = df["split"].values == "probe"

X_train = X[train_mask]
X_val   = X[val_mask]
X_test  = X[test_mask]
X_probe = X[probe_mask]

y_train = y[train_mask]
y_val   = y[val_mask]
y_test  = y[test_mask]
y_probe = y[probe_mask]

print()
print("Dimensioni:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Probe:", X_probe.shape)

print()
print("Numero classi:", len(np.unique(y_train)))

print()
print("Preparazione K = 20 completata.")

## Preparazione dati K = 20...

Dimensioni:
Train: (393783, 48)
Validation: (84383, 48)
Test: (56255, 48)
Probe: (28128, 48)

Numero classi: 20

Preparazione K = 20 completata.


In [ ]:
# REGRESSIONE LOGISTICA — K = 20

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Regressione Logistica K = 20...")

lr_20 = LogisticRegression(
    max_iter=500,
    random_state=42,
    n_jobs=-1
)

lr_20.fit(X_train, y_train)

# Validation
y_val_pred = lr_20.predict(X_val)
f1_val = f1_score(y_val, y_val_pred, average="weighted")

# Test
y_test_pred = lr_20.predict(X_test)
f1_test = f1_score(y_test, y_test_pred, average="weighted")

cm_test = confusion_matrix(y_test, y_test_pred)

print()
print("REGRESSIONE LOGISTICA K = 20")
print("F1 weighted validation:", f1_val)
print("F1 weighted test:", f1_test)

print()
print("Matrice di confusione TEST:")
print(cm_test)

Addestramento Regressione Logistica K = 20...

REGRESSIONE LOGISTICA K = 20
F1 weighted validation: 0.4671231065704996
F1 weighted test: 0.4661355847820558

Matrice di confusione TEST:
[[   0    0    0    0    0    0    0    0  297    0    0  455    0    0
     0    0    0    0    0    0]
 [   0 2970    0    0 3192    0    2    1    1    4    0    0    0    0
     0    0    0    0    0    0]
 [   0   93 4798    0    0    5    0    0 5094    0    0    5    0    0
     0    0  109    0    0    0]
 [   0    0    0    0    0    0    0    0  600    0    0    0    0    0
     0    0    0    0    0    0]
 [   0   12   17    0 4302    0  106   71  473   10   54    0    0    0
     0    0    1    0    0    0]
 [   0    1  259    0    0  147    0    0   89    0    0  367    0    0
     0    0    0    0    0    0]
 [   0   72   33    0   99    0  168   79  182  101    8    0    0    1
     0    4   54    0    3    7]
 [   0    4  309    0  733    1   11  372   17    0   24    0    0    0
     0  

In [ ]:

# RANDOM FOREST — K = 20

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Random Forest K = 20...")

rf_20 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_20.fit(X_train, y_train)

# Validation
y_val_pred_rf = rf_20.predict(X_val)
f1_val_rf = f1_score(y_val, y_val_pred_rf, average="weighted")

# Test
y_test_pred_rf = rf_20.predict(X_test)
f1_test_rf = f1_score(y_test, y_test_pred_rf, average="weighted")

cm_test_rf = confusion_matrix(y_test, y_test_pred_rf)

print()
print("RANDOM FOREST K = 20")
print("F1 weighted validation:", f1_val_rf)
print("F1 weighted test:", f1_test_rf)

print()
print("Matrice di confusione TEST:")
print(cm_test_rf)

Addestramento Random Forest K = 20...

RANDOM FOREST K = 20
F1 weighted validation: 0.8527825262916349
F1 weighted test: 0.8551606401352666

Matrice di confusione TEST:
[[ 478    0    0    0    0    0    0    0    0    0    0  273    0    0
     1    0    0    0    0    0]
 [   0 4840    0    0 1297    0    1   31    0    0    0    0    0    1
     0    0    0    0    0    0]
 [   0    0 9889    0    0    0    0    0  215    0    0    0    0    0
     0    0    0    0    0    0]
 [   0    0    0  600    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [   0  629    0    0 4199    0   11  204    0    0    0    0    0    3
     0    0    0    0    0    0]
 [   4    0  202    0    0  596    0    0    0    0    0   56    0    0
     4    0    0    0    1    0]
 [   0    3    0    0   99    0  705    0    0    1    1    0    0    1
     0    1    0    0    0    0]
 [   0   36    0    0  504    0    2  927    0    0    2    0    0    0
     0    0    0    0   

In [ ]:
import pandas as pd

# SALVATAGGIO RISULTATI E-7

risultati_e7 = pd.DataFrame({
    "K": [2, 2, 20, 20],
    "Modello": [
        "Regressione Logistica",
        "Random Forest",
        "Regressione Logistica",
        "Random Forest"
    ],
    "F1_validation": [
        0.9930409256573689,
        0.999976298601282,
        0.4671231065704996,
        0.8527825262916349
    ],
    "F1_test": [
        0.9934205227454526,
        1.0,
        0.4661355847820558,
        0.8551606401352666
    ]
})

# Salva il file
risultati_e7.to_csv("risultati_E7.csv", index=False)

print("====================================")
print("RISULTATI E-7 SALVATI")
print("====================================")
print(risultati_e7)
print()
print("File creato: risultati_E7.csv")

RISULTATI E-7 SALVATI
    K                Modello  F1_validation   F1_test
0   2  Regressione Logistica       0.993041  0.993421
1   2          Random Forest       0.999976  1.000000
2  20  Regressione Logistica       0.467123  0.466136
3  20          Random Forest       0.852783  0.855161

File creato: risultati_E7.csv


In [ ]:
# E-7bis: controllo della rimozione di dst2src_min_ps

feature_da_rimuovere = "dst2src_min_ps"

indici = [
    i for i, nome in enumerate(feature_names)
    if nome != feature_da_rimuovere
]

feature_names_47 = [feature_names[i] for i in indici]

X_47 = stats_48[:, indici].astype("float32")

print("Numero originale di feature:", len(feature_names))
print("Feature rimossa:", feature_da_rimuovere)
print("Numero di feature dopo la rimozione:", X_47.shape[1])

assert len(feature_names) == 48
assert X_47.shape[1] == 47
assert feature_da_rimuovere not in feature_names_47

Numero originale di feature: 48
Feature rimossa: dst2src_min_ps
Numero di feature dopo la rimozione: 47


In [ ]:
print("Classi in y_train:", np.unique(y_train))
print("Numero di classi:", len(np.unique(y_train)))

print("Classi in y:", np.unique(y))
print("Numero di classi in y:", len(np.unique(y)))

Classi in y_train: ['BitTorrent' 'Cridex' 'FTP' 'Facetime' 'Geodo' 'Gmail' 'Htbot' 'Miuref'
 'MySQL' 'Neris' 'Nsis-ay' 'Outlook' 'SMB' 'Shifu' 'Skype' 'Tinba' 'Virut'
 'Weibo' 'WorldOfWarcraft' 'Zeus']
Numero di classi: 20
Classi in y: ['BitTorrent' 'Cridex' 'FTP' 'Facetime' 'Geodo' 'Gmail' 'Htbot' 'Miuref'
 'MySQL' 'Neris' 'Nsis-ay' 'Outlook' 'SMB' 'Shifu' 'Skype' 'Tinba' 'Virut'
 'Weibo' 'WorldOfWarcraft' 'Zeus']
Numero di classi in y: 20


In [ ]:
print("Colonne disponibili:")
print(df.columns.tolist())

print("\nValori di label_2:")
print(df["label_2"].value_counts(dropna=False))


Colonne disponibili:
['protocol', 'ip_version', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 'src2dst_max_piat_ms', 'dst2src_min_piat_ms', 'dst2src_mean_piat_ms', 'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'bidirectional_syn_packets', 'bidirectional_cwr_packets', 'bidirectional_ece_packets', 'bidirectional_urg_packets', 'bidirectional_ack_packets', 'bidirectional_psh_packets', 'bidirectional_rst_

In [ ]:

# E-7bis — Preparazione dati K = 2 con 47 feature

import numpy as np

y_2 = df["label_2"].to_numpy()
split_2 = df["split"].to_numpy()

train_mask_2 = split_2 == "train"
val_mask_2 = split_2 == "val"
test_mask_2 = split_2 == "test"
probe_mask_2 = split_2 == "probe"

X_train_2_47 = X_47[train_mask_2]
y_train_2 = y_2[train_mask_2]

X_val_2_47 = X_47[val_mask_2]
y_val_2 = y_2[val_mask_2]

X_test_2_47 = X_47[test_mask_2]
y_test_2 = y_2[test_mask_2]

X_probe_2_47 = X_47[probe_mask_2]
y_probe_2 = y_2[probe_mask_2]

print("Classi:", np.unique(y_train_2))
print("Numero di classi:", len(np.unique(y_train_2)))
print("Train:", X_train_2_47.shape)
print("Validation:", X_val_2_47.shape)
print("Test:", X_test_2_47.shape)
print("Probe:", X_probe_2_47.shape)


Classi: ['Benign' 'Malware']
Numero di classi: 2
Train: (393783, 47)
Validation: (84383, 47)
Test: (56255, 47)
Probe: (28128, 47)


In [ ]:

# E-7bis — RANDOM FOREST K = 2, 47 FEATURE

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

rf_2_47 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Addestramento Random Forest K = 2 con 47 feature...")

rf_2_47.fit(X_train_2_47, y_train_2)

y_val_pred_2_47 = rf_2_47.predict(X_val_2_47)
y_test_pred_2_47 = rf_2_47.predict(X_test_2_47)

f1_val_2_47 = f1_score(
    y_val_2, y_val_pred_2_47, average="weighted"
)

f1_test_2_47 = f1_score(
    y_test_2, y_test_pred_2_47, average="weighted"
)

cm_test_2_47 = confusion_matrix(
    y_test_2, y_test_pred_2_47,
    labels=["Benign", "Malware"]
)

print("\nRANDOM FOREST K = 2 — 47 FEATURE")
print("F1 weighted validation:", f1_val_2_47)
print("F1 weighted test:", f1_test_2_47)
print("\nMatrice di confusione TEST (Benign, Malware):")
print(cm_test_2_47)


Addestramento Random Forest K = 2 con 47 feature...

RANDOM FOREST K = 2 — 47 FEATURE
F1 weighted validation: 0.999976298601282
F1 weighted test: 1.0

Matrice di confusione TEST (Benign, Malware):
[[30989     0]
 [    0 25266]]


In [ ]:
print(
    "Predizioni identiche sul test:",
    np.array_equal(y_test_pred_rf, y_test_pred_2_47)
)

Predizioni identiche sul test: False


In [ ]:

# Confronto dettagliato delle predizioni

print("Predizioni originali corrette:",
      np.sum(y_test_pred_rf == y_test))

print("Predizioni con 47 feature corrette:",
      np.sum(y_test_pred_2_47 == y_test_2))

print("Predizioni diverse tra i due modelli:",
      np.sum(y_test_pred_rf != y_test_pred_2_47))

print("Errori modello originale:",
      np.sum(y_test_pred_rf != y_test))

print("Errori modello 47 feature:",
      np.sum(y_test_pred_2_47 != y_test_2))


Predizioni originali corrette: 48404
Predizioni con 47 feature corrette: 56255
Predizioni diverse tra i due modelli: 56255
Errori modello originale: 7851
Errori modello 47 feature: 0


In [ ]:

print("Etichette originali:", np.unique(y_test, return_counts=True))
print("Etichette K=2:", np.unique(y_test_2, return_counts=True))

print("Etichette reali identiche:",
      np.array_equal(y_test, y_test_2))

print("Predizioni originali - prime 20:",
      y_test_pred_rf[:20])

print("Etichette originali - prime 20:",
      y_test[:20])

print("Predizioni 47 feature - prime 20:",
      y_test_pred_2_47[:20])

print("Etichette K=2 - prime 20:",
      y_test_2[:20])


Etichette originali: (array(['BitTorrent', 'Cridex', 'FTP', 'Facetime', 'Geodo', 'Gmail',
       'Htbot', 'Miuref', 'MySQL', 'Neris', 'Nsis-ay', 'Outlook', 'SMB',
       'Shifu', 'Skype', 'Tinba', 'Virut', 'Weibo', 'WorldOfWarcraft',
       'Zeus'], dtype=object), array([  752,  6170, 10104,   600,  5046,   863,   811,  1471,  8609,
        3943,   738,   752,  3894,  1126,   632,  1045,  3741,  3995,
         788,  1175]))
Etichette K=2: (array(['Benign', 'Malware'], dtype=object), array([30989, 25266]))
Etichette reali identiche: False
Predizioni originali - prime 20: ['FTP' 'FTP' 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft'
 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft'
 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft'
 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft'
 'WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarcraft']
Etichette originali - prime 20: ['WorldOfWarcraft' 'WorldOfWarcraft' 'WorldOfWarc

In [ ]:

# Verifica del modello originale a 48 feature per K = 2

X_train_2_48 = stats_48[train_mask_2]
X_val_2_48 = stats_48[val_mask_2]
X_test_2_48 = stats_48[test_mask_2]

rf_2_48_check = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_2_48_check.fit(X_train_2_48, y_train_2)

pred_test_2_48 = rf_2_48_check.predict(X_test_2_48)

f1_test_2_48 = f1_score(
    y_test_2,
    pred_test_2_48,
    average="weighted"
)

print("F1 weighted test — K = 2, 48 feature:", f1_test_2_48)
print("Errori sul test:", np.sum(pred_test_2_48 != y_test_2))


F1 weighted test — K = 2, 48 feature: 1.0
Errori sul test: 0


In [ ]:
# Confronto delle predizioni: 48 contro 47 feature

print(
    "Predizioni identiche:",
    np.array_equal(pred_test_2_48, y_test_pred_2_47)
)

print(
    "Predizioni differenti:",
    np.sum(pred_test_2_48 != y_test_pred_2_47)
)

Predizioni identiche: True
Predizioni differenti: 0


In [ ]:
# Preparazione delle etichette per K = 20

y_20 = df["label_20"].to_numpy()

print("Numero di classi:", len(np.unique(y_20)))
print("Classi:", np.unique(y_20))
print("Dimensione matrice originale:", stats_48.shape)
print("Dimensione matrice ridotta:", X_47.shape)


Numero di classi: 20
Classi: ['BitTorrent' 'Cridex' 'FTP' 'Facetime' 'Geodo' 'Gmail' 'Htbot' 'Miuref'
 'MySQL' 'Neris' 'Nsis-ay' 'Outlook' 'SMB' 'Shifu' 'Skype' 'Tinba' 'Virut'
 'Weibo' 'WorldOfWarcraft' 'Zeus']
Dimensione matrice originale: (562549, 48)
Dimensione matrice ridotta: (562549, 47)


In [ ]:

# E-7bis — Preparazione dati per K = 20

split_20 = df["split"].to_numpy()

train_mask_20 = split_20 == "train"
val_mask_20 = split_20 == "val"
test_mask_20 = split_20 == "test"
probe_mask_20 = split_20 == "probe"

# Dati con 48 feature
X_train_20_48 = stats_48[train_mask_20].astype(np.float32)
X_val_20_48 = stats_48[val_mask_20].astype(np.float32)
X_test_20_48 = stats_48[test_mask_20].astype(np.float32)

# Dati con 47 feature
X_train_20_47 = X_47[train_mask_20].astype(np.float32)
X_val_20_47 = X_47[val_mask_20].astype(np.float32)
X_test_20_47 = X_47[test_mask_20].astype(np.float32)

# Etichette: 20 classi
y_train_20 = y_20[train_mask_20]
y_val_20 = y_20[val_mask_20]
y_test_20 = y_20[test_mask_20]

print("Train:", X_train_20_48.shape, X_train_20_47.shape)
print("Validation:", X_val_20_48.shape, X_val_20_47.shape)
print("Test:", X_test_20_48.shape, X_test_20_47.shape)

print("Classi nel training:", len(np.unique(y_train_20)))
print("Classi nel validation:", len(np.unique(y_val_20)))
print("Classi nel test:", len(np.unique(y_test_20)))

assert X_train_20_48.shape[1] == 48
assert X_train_20_47.shape[1] == 47
assert len(np.unique(y_train_20)) == 20


Train: (393783, 48) (393783, 47)
Validation: (84383, 48) (84383, 47)
Test: (56255, 48) (56255, 47)
Classi nel training: 20
Classi nel validation: 20
Classi nel test: 20


In [ ]:

# E-7bis — Random Forest K = 20 con 48 feature

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

rf_20_48 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Addestramento Random Forest K = 20 con 48 feature...")

rf_20_48.fit(X_train_20_48, y_train_20)

pred_val_20_48 = rf_20_48.predict(X_val_20_48)
pred_test_20_48 = rf_20_48.predict(X_test_20_48)

f1_val_20_48 = f1_score(
    y_val_20,
    pred_val_20_48,
    average="weighted"
)

f1_test_20_48 = f1_score(
    y_test_20,
    pred_test_20_48,
    average="weighted"
)

print("F1 weighted validation:", f1_val_20_48)
print("F1 weighted test:", f1_test_20_48)
print("Errori sul test:", (pred_test_20_48 != y_test_20).sum())


Addestramento Random Forest K = 20 con 48 feature...
F1 weighted validation: 0.8527825262916349
F1 weighted test: 0.8551606401352666
Errori sul test: 7851


In [ ]:

# E-7bis — Random Forest K = 20 con 47 feature

rf_20_47 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Addestramento Random Forest K = 20 con 47 feature...")

rf_20_47.fit(X_train_20_47, y_train_20)

pred_val_20_47 = rf_20_47.predict(X_val_20_47)
pred_test_20_47 = rf_20_47.predict(X_test_20_47)

f1_val_20_47 = f1_score(
    y_val_20,
    pred_val_20_47,
    average="weighted"
)

f1_test_20_47 = f1_score(
    y_test_20,
    pred_test_20_47,
    average="weighted"
)

print("F1 weighted validation:", f1_val_20_47)
print("F1 weighted test:", f1_test_20_47)
print("Errori sul test:", (pred_test_20_47 != y_test_20).sum())


Addestramento Random Forest K = 20 con 47 feature...
F1 weighted validation: 0.8528595705230025
F1 weighted test: 0.8549766388482126
Errori sul test: 7860


In [ ]:

# Confronto delle predizioni K = 20

errori_48 = pred_test_20_48 != y_test_20
errori_47 = pred_test_20_47 != y_test_20

print(
    "Predizioni identiche:",
    np.array_equal(pred_test_20_48, pred_test_20_47)
)

print(
    "Predizioni differenti:",
    np.sum(pred_test_20_48 != pred_test_20_47)
)

print(
    "Esempi corretti con 48 feature ma errati con 47:",
    np.sum((~errori_48) & errori_47)
)

print(
    "Esempi errati con 48 feature ma corretti con 47:",
    np.sum(errori_48 & (~errori_47))
)


Predizioni identiche: False
Predizioni differenti: 41
Esempi corretti con 48 feature ma errati con 47: 22
Esempi errati con 48 feature ma corretti con 47: 13


In [ ]:

# VERIFICA FINALE E-7bis — K = 2
# Ricostruzione dei dati e confronto 48 vs 47 feature

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

# 1. Ricostruzione indipendente di etichette e split
y_check = df["label_2"].to_numpy()
split_check = df["split"].to_numpy()

train_check = split_check == "train"
val_check = split_check == "val"
test_check = split_check == "test"

# 2. Ricostruzione delle matrici
X48_check = stats_48.astype(np.float32)
X47_check = X_47.astype(np.float32)

X48_train = X48_check[train_check]
X48_val = X48_check[val_check]
X48_test = X48_check[test_check]

X47_train = X47_check[train_check]
X47_val = X47_check[val_check]
X47_test = X47_check[test_check]

y_train_check = y_check[train_check]
y_val_check = y_check[val_check]
y_test_check = y_check[test_check]

# 3. Controlli preliminari
assert X48_check.shape == (562549, 48)
assert X47_check.shape == (562549, 47)
assert len(np.unique(y_check)) == 2
assert set(np.unique(y_check)) == {"Benign", "Malware"}
assert len(y_test_check) == 56255

print("Controlli preliminari superati.")
print("Classi:", np.unique(y_check))
print("Train:", X48_train.shape, X47_train.shape)
print("Test:", X48_test.shape, X47_test.shape)

# 4. Addestramento dei due modelli con gli stessi parametri
rf48_check = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf47_check = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf48_check.fit(X48_train, y_train_check)
rf47_check.fit(X47_train, y_train_check)

# 5. Predizioni sullo stesso test set
pred48_check = rf48_check.predict(X48_test)
pred47_check = rf47_check.predict(X47_test)

# 6. Valutazione
print("\n--- RISULTATI K = 2 ---")

print("F1 validation, 48 feature:",
      f1_score(y_val_check, rf48_check.predict(X48_val),
               average="weighted"))

print("F1 validation, 47 feature:",
      f1_score(y_val_check, rf47_check.predict(X47_val),
               average="weighted"))

print("F1 test, 48 feature:",
      f1_score(y_test_check, pred48_check, average="weighted"))

print("F1 test, 47 feature:",
      f1_score(y_test_check, pred47_check, average="weighted"))

print("Errori test, 48 feature:",
      np.sum(pred48_check != y_test_check))

print("Errori test, 47 feature:",
      np.sum(pred47_check != y_test_check))

print("Predizioni identiche:",
      np.array_equal(pred48_check, pred47_check))

print("Predizioni differenti:",
      np.sum(pred48_check != pred47_check))

print("\nMatrice di confusione, 48 feature:")
print(confusion_matrix(
    y_test_check, pred48_check, labels=["Benign", "Malware"]
))

print("\nMatrice di confusione, 47 feature:")
print(confusion_matrix(
    y_test_check, pred47_check, labels=["Benign", "Malware"]
))


Controlli preliminari superati.
Classi: ['Benign' 'Malware']
Train: (393783, 48) (393783, 47)
Test: (56255, 48) (56255, 47)

--- RISULTATI K = 2 ---
F1 validation, 48 feature: 0.999976298601282
F1 validation, 47 feature: 0.999976298601282
F1 test, 48 feature: 1.0
F1 test, 47 feature: 1.0
Errori test, 48 feature: 0
Errori test, 47 feature: 0
Predizioni identiche: True
Predizioni differenti: 0

Matrice di confusione, 48 feature:
[[30989     0]
 [    0 25266]]

Matrice di confusione, 47 feature:
[[30989     0]
 [    0 25266]]


controllo sulla duplicazione di flussi o duplicati tra training e test:

In [ ]:

# Controllo dei duplicati nelle 48 feature numeriche

print("Numero totale di righe:", len(stats_48))

duplicati_48 = len(stats_48) - len(np.unique(stats_48, axis=0))

print("Righe duplicate considerando le 48 feature:", duplicati_48)


Numero totale di righe: 562549
Righe duplicate considerando le 48 feature: 484095


In [ ]:

# Controllo delle combinazioni di feature condivise tra train e test

X_train_dup_check = stats_48[split_check == "train"]
X_test_dup_check = stats_48[split_check == "test"]

# Combinazioni distinte presenti in ciascun insieme
unique_train = np.unique(X_train_dup_check, axis=0)
unique_test = np.unique(X_test_dup_check, axis=0)

# Combinazioni distinte considerando train e test insieme
unique_union = np.unique(
    np.concatenate([unique_train, unique_test], axis=0),
    axis=0
)

# Combinazioni distinte presenti in entrambi gli insiemi
overlap_train_test = (
    len(unique_train) + len(unique_test) - len(unique_union)
)

print("Combinazioni distinte nel training:", len(unique_train))
print("Combinazioni distinte nel test:", len(unique_test))
print("Combinazioni identiche condivise tra train e test:",
      overlap_train_test)
print("Percentuale delle combinazioni distinte del test presenti anche nel train:",
      round(100 * overlap_train_test / len(unique_test), 2), "%")


Combinazioni distinte nel training: 58637
Combinazioni distinte nel test: 12328
Combinazioni identiche condivise tra train e test: 5535
Percentuale delle combinazioni distinte del test presenti anche nel train: 44.9 %


In [ ]:

# Verifica delle etichette condivise tra train e test

def crea_chiavi_righe(matrice):
    matrice = np.ascontiguousarray(matrice)
    return [
        matrice[i].tobytes()
        for i in range(matrice.shape[0])
    ]

X_train_check = stats_48[train_check]
X_test_check = stats_48[test_check]

y_train_label_check = df.loc[train_check, "label_2"].to_numpy()
y_test_label_check = df.loc[test_check, "label_2"].to_numpy()

chiavi_train = crea_chiavi_righe(X_train_check)
chiavi_test = crea_chiavi_righe(X_test_check)

# Associa a ogni combinazione di feature le etichette del training
etichette_per_chiave = {}

for chiave, etichetta in zip(chiavi_train, y_train_label_check):
    if chiave not in etichette_per_chiave:
        etichette_per_chiave[chiave] = set()
    etichette_per_chiave[chiave].add(etichetta)

# Controlla le righe del test
presenti_nel_train = np.array([
    chiave in etichette_per_chiave
    for chiave in chiavi_test
])

stessa_etichetta = np.array([
    chiave in etichette_per_chiave
    and etichetta in etichette_per_chiave[chiave]
    for chiave, etichetta in zip(chiavi_test, y_test_label_check)
])

etichette_conflittuali = sum(
    len(etichette) > 1
    for etichette in etichette_per_chiave.values()
)

print("Righe del test:", len(chiavi_test))
print("Righe con feature già viste nel training:",
      int(presenti_nel_train.sum()))
print("Righe con feature ed etichetta già viste:",
      int(stessa_etichetta.sum()))
print("Combinazioni del training associate a entrambe le etichette:",
      etichette_conflittuali)
print("Percentuale del test con feature già viste:",
      round(100 * presenti_nel_train.mean(), 2), "%")
print("Percentuale del test con feature ed etichetta già viste:",
      round(100 * stessa_etichetta.mean(), 2), "%")



Righe del test: 56255
Righe con feature già viste nel training: 49393
Righe con feature ed etichetta già viste: 49393
Combinazioni del training associate a entrambe le etichette: 3
Percentuale del test con feature già viste: 87.8 %
Percentuale del test con feature ed etichetta già viste: 87.8 %


In [ ]:

# Ricerca delle colonne utili a identificare l'origine dei flussi

print("Colonne del dataset:")
print(df.columns.tolist())

print("\nColonne potenzialmente utili per identificare l'origine:")
parole_chiave = ["pcap", "file", "source", "capture", "flow", "id", "split"]

for colonna in df.columns:
    if any(parola in colonna.lower() for parola in parole_chiave):
        print("-", colonna)


Colonne del dataset:
['protocol', 'ip_version', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 'src2dst_max_piat_ms', 'dst2src_min_piat_ms', 'dst2src_mean_piat_ms', 'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'bidirectional_syn_packets', 'bidirectional_cwr_packets', 'bidirectional_ece_packets', 'bidirectional_urg_packets', 'bidirectional_ack_packets', 'bidirectional_psh_packets', 'bidirectional_rst_

In [ ]:
print(df.columns.tolist())


['protocol', 'ip_version', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 'src2dst_max_piat_ms', 'dst2src_min_piat_ms', 'dst2src_mean_piat_ms', 'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'bidirectional_syn_packets', 'bidirectional_cwr_packets', 'bidirectional_ece_packets', 'bidirectional_urg_packets', 'bidirectional_ack_packets', 'bidirectional_psh_packets', 'bidirectional_rst_packets', 'bidirectio

In [ ]:
print(df["split"].value_counts())

split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64


In [ ]:
print(df["split"].value_counts())
print("\nFile di origine del dataframe:")
print(df.attrs)

split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64

File di origine del dataframe:
{}


In [ ]:
print("Colonne del dataset:")
print(df.columns.tolist())

print("\nPrime righe della colonna split:")
print(df[["split", "label_20", "label_2"]].head(10))

Colonne del dataset:
['protocol', 'ip_version', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 'src2dst_max_piat_ms', 'dst2src_min_piat_ms', 'dst2src_mean_piat_ms', 'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'bidirectional_syn_packets', 'bidirectional_cwr_packets', 'bidirectional_ece_packets', 'bidirectional_urg_packets', 'bidirectional_ack_packets', 'bidirectional_psh_packets', 'bidirectional_rst_

In [ ]:
print("Valori distinti di split:", df["split"].unique())
print("\nDistribuzione delle 20 classi per split:")
print(pd.crosstab(df["split"], df["label_20"]))

Valori distinti di split: ['train' 'val' 'test' 'probe']

Distribuzione delle 20 classi per split:
label_20  BitTorrent  Cridex    FTP  Facetime  Geodo  Gmail  Htbot  Miuref  \
split                                                                        
probe            376    3085   5052       300   2523    431    405     735   
test             752    6170  10104       600   5046    863    811    1471   
train           5261   43189  70725      4200  35325   6041   5675   10294   
val             1128    9255  15156       900   7570   1294   1216    2206   

label_20  MySQL  Neris  Nsis-ay  Outlook    SMB  Shifu  Skype  Tinba  Virut  \
split                                                                         
probe      4305   1972      369      376   1947    563    316    522   1871   
test       8609   3943      738      752   3894   1126    632   1045   3741   
train     60262  27600     5167     5267  27255   7882   4425   7314  26189   
val       12913   5914     1107     1

In [ ]:
print("Colonne che potrebbero identificare l'origine:")
print([
    col for col in df.columns
    if any(parola in col.lower()
           for parola in ["pcap", "file", "source", "capture", "origin"])
])

Colonne che potrebbero identificare l'origine:
[]
